# 01 — Data Quality Check

This notebook performs the initial data-quality assessment of the DWSIM Sobol training dataset.

## Objectives

1. Verify dataset size and structure
2. Inspect data types
3. Check missing values
4. Check NaN and infinite values
5. Detect duplicate samples
6. Verify sampled input ranges
7. Verify derived feed temperature
8. Validate product compositions
9. Validate condenser and reboiler duties
10. Check physical consistency
11. Report convergence statistics

### Important

The 500-point LHS dataset is the final holdout dataset.

It is intentionally NOT loaded or inspected for model-development decisions in this notebook.

In [2]:
import os
import math
import numpy as np
import pandas as pd

from IPython.display import display

## 1. Load Training Data

Only the Sobol training dataset is used in this notebook.

The non-converged cases are retained separately for convergence analysis.

In [3]:
df = pd.read_csv("../data/02_raw_train_sobol/dataset_2560_converged.csv")

df_failed = pd.read_csv("../data/02_raw_train_sobol/dataset_2560_not_converged.csv")

print("Converged dataset shape:", df.shape)

print("Non-converged dataset shape:", df_failed.shape)

Converged dataset shape: (2524, 25)
Non-converged dataset shape: (36, 25)


## 2. Dataset Overview

First inspect the number of observations, variables, and column names.

In [4]:
print("Number of converged samples:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

Number of converged samples: 2524
Number of columns: 25

Columns:
 1. sobol_index
 2. batch
 3. pressure_atm
 4. requested_vapor_fraction
 5. benzene_feed_fraction
 6. toluene_feed_fraction
 7. stages
 8. feed_stage
 9. feed_stage_fraction
10. reflux_ratio
11. bottoms_fraction
12. feed_flow_kmol_h
13. feed_temperature_C
14. x_D_benzene
15. x_B_benzene
16. Q_C
17. Q_R
18. dwsim_solved
19. column_calculated
20. column_error
21. output_values_valid
22. composition_valid
23. temperature_valid
24. case_valid
25. error_message


In [5]:
display(df.head())

,sobol_index,batch,pressure_atm,requested_vapor_fraction,benzene_feed_fraction,toluene_feed_fraction,stages,feed_stage,feed_stage_fraction,reflux_ratio,...,Q_C,Q_R,dwsim_solved,column_calculated,column_error,output_values_valid,composition_valid,temperature_valid,case_valid,error_message
0,1,2048,1.431029,0.244310,0.622565,0.377435,11,5,0.454545,3.971597,...,1894.685132,1902.616504,True,True,NaN,True,True,True,True,NaN
1,2,2048,1.882465,0.047053,0.430769,0.569231,29,15,0.517241,2.082134,...,1338.133793,1354.195832,True,True,NaN,True,True,True,True,NaN
2,3,2048,1.703318,0.222960,0.585932,0.414068,17,10,0.588235,1.859158,...,1018.058123,1027.198908,True,True,NaN,True,True,True,True,NaN
3,4,2048,1.233188,0.082173,0.367573,0.632427,23,8,0.347826,3.266472,...,2246.606897,2256.197475,True,True,NaN,True,True,True,True,NaN
4,5,2048,1.103865,0.158570,0.493004,0.506996,22,8,0.363636,4.327704,...,1806.772138,1820.154981,True,True,NaN,True,True,True,True,NaN


In [6]:
display(df.head())

,sobol_index,batch,pressure_atm,requested_vapor_fraction,benzene_feed_fraction,toluene_feed_fraction,stages,feed_stage,feed_stage_fraction,reflux_ratio,...,Q_C,Q_R,dwsim_solved,column_calculated,column_error,output_values_valid,composition_valid,temperature_valid,case_valid,error_message
0,1,2048,1.431029,0.244310,0.622565,0.377435,11,5,0.454545,3.971597,...,1894.685132,1902.616504,True,True,NaN,True,True,True,True,NaN
1,2,2048,1.882465,0.047053,0.430769,0.569231,29,15,0.517241,2.082134,...,1338.133793,1354.195832,True,True,NaN,True,True,True,True,NaN
2,3,2048,1.703318,0.222960,0.585932,0.414068,17,10,0.588235,1.859158,...,1018.058123,1027.198908,True,True,NaN,True,True,True,True,NaN
3,4,2048,1.233188,0.082173,0.367573,0.632427,23,8,0.347826,3.266472,...,2246.606897,2256.197475,True,True,NaN,True,True,True,True,NaN
4,5,2048,1.103865,0.158570,0.493004,0.506996,22,8,0.363636,4.327704,...,1806.772138,1820.154981,True,True,NaN,True,True,True,True,NaN


## 3. Data Types

Check whether each variable has the expected numerical representation.

In [7]:
dtype_table = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing": df.isna().sum().values
})

display(dtype_table)

,column,dtype,missing
0,sobol_index,int64,0
1,batch,int64,0
2,pressure_atm,float64,0
3,requested_vapor_fraction,float64,0
4,benzene_feed_fraction,float64,0
5,toluene_feed_fraction,float64,0
6,stages,int64,0
7,feed_stage,int64,0
8,feed_stage_fraction,float64,0
9,reflux_ratio,float64,0


## 4. Missing Values

I will check for missing values in the dataset.

The `error_message` and `column_error` columns are diagnostic columns used during DWSIM simulation. They are empty for successful cases and will not be required for machine learning.

I will keep the raw dataset unchanged and remove these diagnostic columns later when creating the cleaned dataset.

In [8]:
missing = df.isna().sum()

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (
            missing / len(df) * 100
    )
})

display(
    missing_table.sort_values(
        "missing_count",
        ascending=False
    )
)

,missing_count,missing_percent
error_message,2524,100.0
column_error,2524,100.0
pressure_atm,0,0.0
batch,0,0.0
benzene_feed_fraction,0,0.0
toluene_feed_fraction,0,0.0
stages,0,0.0
requested_vapor_fraction,0,0.0
sobol_index,0,0.0
feed_stage_fraction,0,0.0


In [9]:
total_missing = int(
    df.isna().sum().sum()
)
print("Total missing values:", total_missing)

Total missing values: 5048


## 5. Duplicate Input Samples

I will check whether any two cases contain the same sampled input values.

Duplicate input combinations should not be present in the Sobol dataset.

In [10]:
input_columns = [
    "pressure_atm",
    "requested_vapor_fraction",
    "benzene_feed_fraction",
    "stages",
    "feed_stage_fraction",
    "reflux_ratio",
    "bottoms_fraction"
]

duplicate_count = df.duplicated(
    subset=input_columns
).sum()

print(
    "Duplicate input samples:",
    duplicate_count
)

Duplicate input samples: 0


## 6. Input Range Check

I will check whether all the sampled input values are within the ranges defined for the DWSIM simulations.

Each input should remain within its specified minimum and maximum value.

In [11]:
ranges = {
    "pressure_atm": (1.0, 2.0),
    "requested_vapor_fraction": (0.0, 0.30),
    "benzene_feed_fraction": (0.30, 0.70),
    "stages": (10, 30),
    "feed_stage_fraction": (0.30, 0.70),
    "reflux_ratio": (1.2, 4.5),
    "bottoms_fraction": (0.40, 0.60)
}

for column, (minimum, maximum) in ranges.items():

    outside = (
            (df[column] < minimum) |
            (df[column] > maximum)
    ).sum()

    print(
        f"{column}: {outside} values outside range"
    )

pressure_atm: 0 values outside range
requested_vapor_fraction: 0 values outside range
benzene_feed_fraction: 0 values outside range
stages: 0 values outside range
feed_stage_fraction: 68 values outside range
reflux_ratio: 0 values outside range
bottoms_fraction: 0 values outside range


## 7. Inspecting Feed-Stage Fraction

68 cases were found outside the specified feed-stage fraction range.

Before removing any cases, I will inspect these values and compare them with the corresponding number of stages and feed stage.

In [12]:
invalid_feed_stage = df[
    (df["feed_stage_fraction"] < 0.30) |
    (df["feed_stage_fraction"] > 0.70)
    ]

print(
    "Number of cases outside range:",
    len(invalid_feed_stage)
)

display(
    invalid_feed_stage[
        [
            "stages",
            "feed_stage",
            "feed_stage_fraction"
        ]
    ].head(20)
)

Number of cases outside range: 68


,stages,feed_stage,feed_stage_fraction
75,17,5,0.294118
106,21,6,0.285714
166,24,7,0.291667
196,17,12,0.705882
236,27,8,0.296296
261,18,13,0.722222
300,28,8,0.285714
331,17,5,0.294118
345,14,4,0.285714
354,27,19,0.703704


## 7. Feed Stage Check

The feed stage is converted from the sampled feed-stage fraction into an integer stage number before running DWSIM.

I will check whether every feed stage is a valid stage in the column.

The feed stage should satisfy:

1 <= feed_stage <= stages

In [13]:
invalid_feed_stage = df[
    (df["feed_stage"] < 1) |
    (df["feed_stage"] > df["stages"])
    ]

print(
    "Invalid feed stages:",
    len(invalid_feed_stage)
)

Invalid feed stages: 0


## 8. Feed Flow Check

The feed flow is fixed at 100 kmol/h for all DWSIM simulations.

I will check whether all cases contain the expected feed flow.

In [14]:
print(
    "Unique feed flow values:"
)

print(
    df["feed_flow_kmol_h"].unique()
)

invalid_feed_flow = (
        df["feed_flow_kmol_h"] != 100.0
).sum()

print(
    "Invalid feed flow values:",
    invalid_feed_flow
)

Unique feed flow values:
[100.]
Invalid feed flow values: 0


## 9. Feed Composition Check

The feed contains only Benzene and Toluene.

Therefore, the Benzene and Toluene mole fractions should add up to 1 for every case.

In [15]:
composition_sum = (
        df["benzene_feed_fraction"]
        + df["toluene_feed_fraction"]
)

composition_error = np.abs(
    composition_sum - 1.0
)

print(
    "Maximum composition error:",
    composition_error.max()
)

print(
    "Cases with composition error:",
    (composition_error > 1e-10).sum()
)

Maximum composition error: 1.1102230246251565e-16
Cases with composition error: 0


## 10. Derived Feed Temperature Check

The feed temperature was not sampled as an input.

It was derived by DWSIM from the specified pressure and vapor fraction.

I will check whether all derived feed temperatures are valid numerical values.

In [16]:
temperature = df["feed_temperature_C"]

print(
    "Invalid temperature values:",
    (~np.isfinite(temperature)).sum()
)

print(
    "Minimum temperature:",
    temperature.min()
)

print(
    "Maximum temperature:",
    temperature.max()
)

Invalid temperature values: 0
Minimum temperature: 86.86673664297797
Maximum temperature: 124.21248127744616


## 11. Product Composition Check

The Benzene mole fraction in both the distillate and bottoms streams should be between 0 and 1.

I will check whether any case contains a product composition outside this physical range.

In [17]:
invalid_xD = (
        (df["x_D_benzene"] < 0) |
        (df["x_D_benzene"] > 1)
).sum()

invalid_xB = (
        (df["x_B_benzene"] < 0) |
        (df["x_B_benzene"] > 1)
).sum()

print(
    "Invalid x_D values:",
    invalid_xD
)

print(
    "Invalid x_B values:",
    invalid_xB
)

Invalid x_D values: 0
Invalid x_B values: 0


## 12. Energy Duty Check

The condenser duty and reboiler duty are important DWSIM outputs.

I will check whether any case contains NaN, infinite, or zero duty values.

In [18]:
for column in ["Q_C", "Q_R"]:

    values = df[column]

    print(f"\n{column}")

    print(
        "Invalid values:",
        (~np.isfinite(values)).sum()
    )

    print(
        "Zero values:",
        (values == 0).sum()
    )

    print(
        "Minimum:",
        values.min()
    )

    print(
        "Maximum:",
        values.max()
    )


Q_C
Invalid values: 0
Zero values: 0
Minimum: 748.729566508234
Maximum: 2828.747576937871

Q_R
Invalid values: 0
Zero values: 0
Minimum: 753.1182892286442
Maximum: 2838.457528093876


## 13. DWSIM Calculation Status

I will check whether all cases in the converged dataset were successfully solved by DWSIM and whether the distillation column was successfully calculated.

In [19]:
print(
    "DWSIM solved:",
    df["dwsim_solved"].value_counts()
)

print(
    "\nColumn calculated:",
    df["column_calculated"].value_counts()
)

DWSIM solved: dwsim_solved
True    2524
Name: count, dtype: int64

Column calculated: column_calculated
True    2524
Name: count, dtype: int64


## 14. Final Case Validity Check

I will check the final validation flag stored during the DWSIM simulation.

Every case in the converged dataset should have passed all the validation checks used during data generation.

In [20]:
print(
    df["case_valid"].value_counts()
)

case_valid
True    2524
Name: count, dtype: int64


## 15. Output Validation Check

I will check the validation flags recorded during the DWSIM simulation for the main outputs.

All converged cases should have valid output values, valid compositions, and valid derived feed temperatures.

In [21]:
validation_columns = [
    "output_values_valid",
    "composition_valid",
    "temperature_valid"
]

for column in validation_columns:
    print(
        f"\n{column}:"
    )
    print(
        df[column].value_counts()
    )


output_values_valid:
output_values_valid
True    2524
Name: count, dtype: int64

composition_valid:
composition_valid
True    2524
Name: count, dtype: int64

temperature_valid:
temperature_valid
True    2524
Name: count, dtype: int64


## 16. Checking Non-Converged Cases

The non-converged DWSIM cases are kept separately from the valid training data.

I will check the number of failed cases and the recorded error messages to understand why these simulations failed.

In [22]:
print(
    "Number of non-converged cases:",
    len(df_failed)
)

print("\nError messages:")

display(
    df_failed["error_message"]
    .value_counts()
)

print(
    df_failed["error_message"].to_list()
)

Number of non-converged cases: 36

Error messages:


error_message
DWSIM did not converge.    36
Name: count, dtype: int64

['DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.', 'DWSIM did not converge.']


## 17. DWSIM Convergence Rate

I will calculate the percentage of simulations that successfully converged.

The converged and non-converged cases are kept separately.

In [23]:
converged_count = len(df)
not_converged_count = len(df_failed)

total_count = (
        converged_count +
        not_converged_count
)

convergence_rate = (
        converged_count /
        total_count *
        100
)

print("Total cases:", total_count)
print("Converged cases:", converged_count)
print("Non-converged cases:", not_converged_count)
print(
    f"Convergence rate: {convergence_rate:.2f}%"
)

Total cases: 2560
Converged cases: 2524
Non-converged cases: 36
Convergence rate: 98.59%
